In [ ]:
!pip install transformers datasets accelerate -U -q
!pip install sentencepiece evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 93.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.0/527.0 kB 45.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 15.5 MB/s eta 0:00:00


In [29]:
!pip install --upgrade transformers huggingface_hub accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 642.6/642.6 kB 22.1 MB/s eta 0:00:00
  Attempting uninstall: huggingface_hub
    Found existing installation: huggingface_hub 1.10.1
    Uninstalling huggingface_hub-1.10.1:
      Successfully uninstalled huggingface_hub-1.10.1


In [31]:
import zipfile
import os
import gc
import torch
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import zipfile
from torch.utils.data import DataLoader, Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
from sklearn.metrics import classification_report, confusion_matrix, f1_score
import pandas as pd
import pyarrow.parquet as pq
from datasets import load_from_disk

In [29]:
DRIVE_DIR = Path('/content/drive/MyDrive/DM/dataset')
TRAIN_ZIP = DRIVE_DIR / 'train_data.parquet.zip'
VAL_ZIP = DRIVE_DIR / 'val_data.parquet.zip'

OUTPUT_DIR = Path('/content/drive/MyDrive/DM/outputs_dl')
FIGURES_DIR = Path('/content/drive/MyDrive/DM/figures_dl')

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
EXTRACT_TRAIN_DIR = Path('/content/train_data')
EXTRACT_VAL_DIR = Path('/content/val_data')
OUTPUT_DIR = Path('/content/drive/MyDrive/outputs_dl')

for p in [EXTRACT_TRAIN_DIR, EXTRACT_VAL_DIR, OUTPUT_DIR]:
    p.mkdir(parents=True, exist_ok=True)

LABEL_NAMES = ['negative', 'neutral', 'positive']

In [ ]:
import evaluate
import numpy as np

metric = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    # Lấy chỉ số có xác suất cao nhất làm dự đoán
    predictions = np.argmax(logits, axis=-1)

    # Tính toán độ chính xác
    return metric.compute(predictions=predictions, references=labels)

# **Giải nén train val và tạo data map** ( không cần chạy khi đã có dữ liệu map)

In [18]:
def extract_zip(zip_path, target_dir):
    if zip_path.exists():
        print(f"Dang giai nen {zip_path.name}...")
        with zipfile.ZipFile(zip_path, 'r') as zip_ref:
            zip_ref.extractall(target_dir)
        print(f"Xong: {target_dir}")
    else:
        print(f"Loi: Khong tim thay {zip_path}")

extract_zip(TRAIN_ZIP, EXTRACT_TRAIN_DIR)
extract_zip(VAL_ZIP, EXTRACT_VAL_DIR)




Dang giai nen train_data.parquet.zip...
Xong: /content/train_data
Dang giai nen val_data.parquet.zip...
Xong: /content/val_data


In [19]:
from datasets import load_dataset

# 1. Đường dẫn file
data_files = {
    "train": str(EXTRACT_TRAIN_DIR / "**/*.parquet"),
    "test": str(EXTRACT_VAL_DIR / "**/*.parquet")
}

# 2. Nạp TOÀN BỘ nhưng CHỈ lấy 2 cột text và label
raw_datasets = load_dataset(
    "parquet",
    data_files=data_files,
    columns=['text', 'label']
)

# Sử dụng toàn bộ dữ liệu không qua select
train_dataset = raw_datasets["train"].shuffle(seed=42)
val_dataset = raw_datasets["test"]

print(f"--- Đã nạp xong TOÀN BỘ dữ liệu ---")
print(f"Tổng số mẫu Train: {len(train_dataset)}")
print(f"Tổng số mẫu Val: {len(val_dataset)}")

Generating train split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

--- Đã nạp xong TOÀN BỘ dữ liệu ---
Tổng số mẫu Train: 16241239
Tổng số mẫu Val: 3479704


In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
import multiprocessing

cores = multiprocessing.cpu_count()

model_name = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

tokenizer.add_special_tokens({'pad_token': '[PAD]'})

def tokenize_function(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True, max_length=128)

cores = 2

tokenized_train = train_dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=['text'],
    num_proc=cores,
    batch_size=1000
)
tokenized_val = val_dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=['text'],
    num_proc=cores,
    batch_size=1000
)



In [ ]:
# # Lưu vào thư mục tạm trên Colab trước (để zip cho nhanh)
# save_path_local = "/content/tokenized_data_all"
# tokenized_train.save_to_disk(f"{save_path_local}/train")
# tokenized_val.save_to_disk(f"{save_path_local}/val")

# print("Đã lưu xong vào ổ đĩa tạm của Colab.")

In [ ]:
# import shutil
# # Nén thư mục thành file .zip
# shutil.make_archive("/content/tokenized_data_all", 'zip', save_path_local)

In [ ]:
#!cp /content/tokenized_data_all.zip /content/drive/MyDrive/DM/dataset

# đọc file data map từ drive

In [41]:
zip_map_path = '/content/drive/MyDrive/DM/dataset/tokenized_data_all.zip'
!unzip -q "{zip_map_path}" -d /content/dataset_local

tokenized_train = load_from_disk(f"{extract_folder}/train")
tokenized_val = load_from_disk(f"{extract_folder}/val")

print("Dữ liệu đã nạp thành công!")
print(f"Số lượng mẫu huấn luyện: {len(tokenized_train)}")

Loading dataset from disk:   0%|          | 0/22 [00:00<?, ?it/s]

Dữ liệu đã nạp thành công!
Số lượng mẫu huấn luyện: 16241239


# Train

In [44]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
import multiprocessing

cores = multiprocessing.cpu_count()

model_name = "distilbert-base-uncased"
# Huấn luyện
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=3).to("cuda")

training_args = TrainingArguments(
    output_dir='/content/results',
    dataloader_num_workers=2,
    num_train_epochs=1,
    per_device_train_batch_size=384,

    learning_rate=8e-5,               # Khuyến nghị cho batch size 384
    lr_scheduler_type="linear",       # Giảm dần lr theo thời gian
    warmup_steps=5000,                # Tăng lr từ từ ở giai đoạn đầu để ổn định

    fp16=True,
    logging_steps=2000,
    eval_strategy="steps",
    eval_steps=15000,
    save_strategy="no",
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    compute_metrics=compute_metrics
)

print("Bắt đầu huấn luyện TOÀN BỘ dữ liệu...")
trainer.train()

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Bắt đầu huấn luyện TOÀN BỘ dữ liệu...


Step,Training Loss,Validation Loss


KeyboardInterrupt: 

In [ ]:
trainer.evaluate()

In [ ]:
model_path =  "/content/drive/MyDrive/DM/outputs_dl/DiBert"
trainer.save_model(model_path)
tokenizer.save_pretrained(model_path)
print("Da luu mo hinh thanh cong!")

In [ ]:
from transformers import pipeline
classifier = pipeline("sentiment-analysis", model=model_path, tokenizer=model_path, device=0)
print(classifier("this company is bad, but i love it!"))